In [138]:
import os
import torch
import numpy as np
import lightgbm as lgb
import warnings

# Quantile Regression

In [ ]:
warnings.filterwarnings('ignore')
def calculate_interval_score(y_true, lower, upper, alpha):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2.0 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2.0 / alpha) * (y_true - upper), 0)
    return np.mean(width + penalty_lower + penalty_upper)

def calculate_cwc(picp, mpiw, y_trues, t_val, eta=50):
    y_max, y_min = y_trues.max(), y_trues.min()
    range_y = y_max - y_min if (y_max - y_min) > 0 else 1.0
    nmpiw = mpiw / range_y
    gamma = 1 if picp < t_val else 0
    penalty = gamma * np.exp(-eta * (picp - t_val))
    return nmpiw + penalty

def run_quantile_baseline(split_dir="split_embeddings"):
    print("Running Baseline 1: Pure Quantile Regression (LightGBM)...\n")
    
    category_t_map = {
        "BIKES": 0.95, "BOOKS": 0.99, "CARS": 0.99, "CYCLE": 0.92,
        "FLAT": 0.95, "FRIDGES": 0.99, "GAMES": 0.95, "GAMESENTERTAINMENT": 0.98,
        "LAPTOP": 0.99, "PHONES": 0.99, "PRINTER": 0.98,
        "TV": 0.99, "WASHINGMACHINE": 0.99
    }
    
    train_files = [f for f in os.listdir(os.path.join(split_dir, 'train')) if f.endswith('.pt')]
    
    for file in train_files:
        cat_name = file.replace('combined_', '').replace('.pt', '').upper()
        train_path = os.path.join(split_dir, 'train', file)
        test_path = os.path.join(split_dir, 'test', file)
        
        if not os.path.exists(test_path): continue
            
        t_val = category_t_map.get(cat_name, 0.95)
        alpha = 1.0 - t_val
        
        print(f"\nQUANTILE GBM: {cat_name}\n{'-'*50}")
        
        raw_train_data = torch.load(train_path, weights_only=False)
        raw_test_data = torch.load(test_path, weights_only=False)
        
        cat_mu = raw_train_data['log_prices'].mean().item()
        cat_sigma = raw_train_data['log_prices'].std().item()
        
        X_train = raw_train_data['embeddings'].numpy()
        y_train = ((raw_train_data['log_prices'] - cat_mu) / cat_sigma).numpy()
        
        X_test = raw_test_data['embeddings'].numpy()
        y_test_real = np.exp(raw_test_data['log_prices'].numpy())
        
        lower_quantile = alpha / 2.0
        upper_quantile = 1.0 - (alpha / 2.0)
        
        model_lower = lgb.LGBMRegressor(objective='quantile', alpha=lower_quantile, n_estimators=150, verbose=-1, random_state=42)
        model_upper = lgb.LGBMRegressor(objective='quantile', alpha=upper_quantile, n_estimators=150, verbose=-1, random_state=42)
        
        model_lower.fit(X_train, y_train)
        model_upper.fit(X_train, y_train)
        
        test_preds_lower = model_lower.predict(X_test)
        test_preds_upper = model_upper.predict(X_test)
        
        l_real = np.exp(test_preds_lower * cat_sigma + cat_mu)
        u_real = np.exp(test_preds_upper * cat_sigma + cat_mu)
        
        picp = np.mean((y_test_real >= l_real) & (y_test_real <= u_real))
        mpiw = np.mean(u_real - l_real)
        is_score = calculate_interval_score(y_test_real, l_real, u_real, alpha=alpha)
        cwc_score = calculate_cwc(picp, mpiw, y_test_real, t_val=t_val)
        
        print(f"PICP (Coverage):   {picp:.4f}")
        print(f"MPIW (Width):      ₹{mpiw:,.0f}")
        print(f"CWC Score:         {cwc_score:.4f}")
        print(f"Interval Score:    {is_score:,.0f}\n")

run_quantile_baseline()


Running Baseline 1: Pure Quantile Regression (LightGBM)...


QUANTILE GBM: BIKES
--------------------------------------------------
PICP (Coverage):   0.9773
MPIW (Width):      ₹237,118
CWC Score:         0.3764
Interval Score:    239,650


QUANTILE GBM: BOOKS
--------------------------------------------------
PICP (Coverage):   0.9565
MPIW (Width):      ₹12,103
CWC Score:         6.1442
Interval Score:    37,691


QUANTILE GBM: CARS
--------------------------------------------------
PICP (Coverage):   0.8710
MPIW (Width):      ₹6,214,785
CWC Score:         386.3932
Interval Score:    7,291,044


QUANTILE GBM: CYCLE
--------------------------------------------------
PICP (Coverage):   0.9310
MPIW (Width):      ₹27,316
CWC Score:         0.3436
Interval Score:    40,645


QUANTILE GBM: FLAT
--------------------------------------------------
PICP (Coverage):   0.9608
MPIW (Width):      ₹16,422,984
CWC Score:         0.8644
Interval Score:    18,696,610


QUANTILE GBM: FRIDGES
-----------

In [80]:
run_quantile_baseline()

Running Baseline 1: Pure Quantile Regression (LightGBM)...


QUANTILE GBM: BIKES
--------------------------------------------------
PICP (Coverage):   0.9773
MPIW (Width):      ₹237,118
CWC Score:         0.3764
Interval Score:    239,650


QUANTILE GBM: BOOKS
--------------------------------------------------
PICP (Coverage):   0.9565
MPIW (Width):      ₹12,103
CWC Score:         6.1442
Interval Score:    37,691


QUANTILE GBM: CARS
--------------------------------------------------
PICP (Coverage):   0.8710
MPIW (Width):      ₹6,214,785
CWC Score:         386.3932
Interval Score:    7,291,044


QUANTILE GBM: CYCLE
--------------------------------------------------
PICP (Coverage):   0.9310
MPIW (Width):      ₹27,316
CWC Score:         0.3436
Interval Score:    40,645


QUANTILE GBM: FLAT
--------------------------------------------------
PICP (Coverage):   0.9608
MPIW (Width):      ₹16,422,984
CWC Score:         0.8644
Interval Score:    18,696,610


QUANTILE GBM: FRIDGES
-----------

# Tube loss Baseline

In [139]:
import os
import torch
import numpy as np
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

In [166]:
class ScaledRFPDataset(Dataset):
    def __init__(self, pt_file_path, mu, sigma):
        data = torch.load(pt_file_path, weights_only=False)
        self.embeddings = data['embeddings']
        self.indices = data.get('dataframe_indices', list(range(len(self.embeddings)))) 
        self.targets = (data['log_prices'] - mu) / sigma
        
    def __len__(self): return len(self.embeddings)
    def __getitem__(self, idx):
        return self.embeddings[idx], torch.tensor(self.targets[idx], dtype=torch.float32)


def tube_loss(y, mu1, mu2, t=0.95, r=0.5, delta=0.0):
    lower = torch.min(mu1, mu2)
    upper = torch.max(mu1, mu2)
    
    loss = torch.zeros_like(y)
    loss[y > upper] = t * (y[y > upper] - upper[y > upper])
    loss[y < lower] = t * (lower[y < lower] - y[y < lower])

    mid = r * upper + (1 - r) * lower
    mask = (y >= lower) & (y <= upper)
    loss[mask & (y >= mid)] = (1 - t) * (upper[mask & (y >= mid)] - y[mask & (y >= mid)])
    loss[mask & (y < mid)]  = (1 - t) * (y[mask & (y < mid)] - lower[mask & (y < mid)])
    
    base_loss = loss.mean()
    width_penalty = torch.abs(upper - lower).mean()
    
    return base_loss + (delta * width_penalty)


def calculate_interval_score(y_true, lower, upper, alpha):
    width = upper - lower
    penalty_lower = torch.where(y_true < lower, (2.0 / alpha) * (lower - y_true), torch.zeros_like(width))
    penalty_upper = torch.where(y_true > upper, (2.0 / alpha) * (y_true - upper), torch.zeros_like(width))
    return (width + penalty_lower + penalty_upper).mean().item()

def calculate_cwc(picp, mpiw, y_trues, t_val, eta=50):
    y_max, y_min = y_trues.max().item(), y_trues.min().item()
    range_y = y_max - y_min if (y_max - y_min) > 0 else 1.0
    nmpiw = mpiw / range_y
    gamma = 1 if picp < t_val else 0
    penalty = gamma * np.exp(-eta * (picp - t_val))
    return nmpiw + penalty


class SimpleCategoryMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(1536, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(512, 256), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(256, 2)
        )
    def forward(self, x): return self.head(x.float())


def run_tube_loss_baseline(split_dir="split_embeddings", epochs=15):
    print("Running Baseline 2: Tube Loss (LUBE-Style)...\n")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    category_t_map = {
        "BIKES": 0.95, "BOOKS": 0.98, "CARS": 0.96, "CYCLE": 0.95,
        "FLAT": 0.96, "FRIDGES": 0.96, "GAMES": 0.96, "GAMESENTERTAINMENT": 0.98,
        "LAPTOP": 0.95, "PHONES": 0.98, "PRINTER": 0.98,
        "TV": 0.99, "WASHINGMACHINE": 0.98
    }
    
    train_files = [f for f in os.listdir(os.path.join(split_dir, 'train')) if f.endswith('.pt')]
    
    for file in train_files:
        cat_name = file.replace('combined_', '').replace('.pt', '').upper()
        train_path = os.path.join(split_dir, 'train', file)
        test_path = os.path.join(split_dir, 'test', file)
        
        if not os.path.exists(test_path): continue
            
        t_val = category_t_map.get(cat_name, 0.95)
        alpha = 1.0 - t_val
        
        print(f"TUBE LOSS BASELINE: {cat_name}\n{'-'*50}")
        
        # Load Data
        raw_train_data = torch.load(train_path, weights_only=False)
        cat_mu = raw_train_data['log_prices'].mean().item()
        cat_sigma = raw_train_data['log_prices'].std().item()
        
        train_dataset = ScaledRFPDataset(train_path, cat_mu, cat_sigma)
        test_dataset = ScaledRFPDataset(test_path, cat_mu, cat_sigma)
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
        
        # Initialize
        model = SimpleCategoryMLP().to(device)
        optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        # Train (DELTA = 0.0 for initial training as per paper)
        model.train()
        for epoch in range(epochs):
            for x, y in train_loader:
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad()
                out = model(x)
                loss = tube_loss(y, out[:,0], out[:,1], t=t_val, r=0.5, delta=0.0)
                loss.backward()
                optimizer.step()

        # Evaluate
        model.eval()
        lowers, uppers, ys = [], [], []
        
        with torch.no_grad():
            for x_batch, y_batch in test_loader:
                x_device = x_batch.to(device)
                out = model(x_device)
                l, u = torch.min(out[:,0], out[:,1]), torch.max(out[:,0], out[:,1])
                
                l_real = torch.exp(l * cat_sigma + cat_mu)
                u_real = torch.exp(u * cat_sigma + cat_mu)
                y_true = torch.exp(y_batch.to(device) * cat_sigma + cat_mu)
                
                lowers.append(l_real); uppers.append(u_real); ys.append(y_true)
                
        l_preds = torch.cat(lowers)
        u_preds = torch.cat(uppers)
        y_trues = torch.cat(ys)
        
        picp = ((y_trues >= l_preds) & (y_trues <= u_preds)).float().mean().item()
        mpiw = (u_preds - l_preds).mean().item()
        is_score = calculate_interval_score(y_trues, l_preds, u_preds, alpha=alpha)
        cwc_score = calculate_cwc(picp, mpiw, y_trues, t_val=t_val)
        
        print(f"PICP (Coverage):   {picp:.4f}")
        print(f"MPIW (Width):      ₹{mpiw:,.0f}")
        print(f"CWC Score:         {cwc_score:.4f}")
        print(f"Interval Score:    {is_score:,.0f}\n")

torch.manual_seed(42)


In [129]:
run_tube_loss_baseline(epochs=15)

Running Baseline 2: Tube Loss (LUBE-Style)...

TUBE LOSS BASELINE: BIKES
--------------------------------------------------
PICP (Coverage):   1.0000
MPIW (Width):      ₹235,194
CWC Score:         0.3733
Interval Score:    235,194

TUBE LOSS BASELINE: BOOKS
--------------------------------------------------
PICP (Coverage):   0.9565
MPIW (Width):      ₹7,146
CWC Score:         3.7136
Interval Score:    36,025

TUBE LOSS BASELINE: CARS
--------------------------------------------------
PICP (Coverage):   1.0000
MPIW (Width):      ₹6,996,750
CWC Score:         2.2746
Interval Score:    6,996,750

TUBE LOSS BASELINE: CYCLE
--------------------------------------------------
PICP (Coverage):   0.9655
MPIW (Width):      ₹446,230
CWC Score:         5.6130
Interval Score:    446,677

TUBE LOSS BASELINE: FLAT
--------------------------------------------------
PICP (Coverage):   0.9804
MPIW (Width):      ₹21,371,280
CWC Score:         1.1248
Interval Score:    24,895,990

TUBE LOSS BASELINE: FRI

In [130]:
run_tube_loss_baseline(epochs=15)

Running Baseline 2: Tube Loss (LUBE-Style)...

TUBE LOSS BASELINE: BIKES
--------------------------------------------------
PICP (Coverage):   1.0000
MPIW (Width):      ₹331,379
CWC Score:         0.5260
Interval Score:    331,379

TUBE LOSS BASELINE: BOOKS
--------------------------------------------------
PICP (Coverage):   0.8478
MPIW (Width):      ₹6,349
CWC Score:         741.9416
Interval Score:    51,852

TUBE LOSS BASELINE: CARS
--------------------------------------------------
PICP (Coverage):   0.9355
MPIW (Width):      ₹21,771,068
CWC Score:         10.4846
Interval Score:    21,864,452

TUBE LOSS BASELINE: CYCLE
--------------------------------------------------
PICP (Coverage):   1.0000
MPIW (Width):      ₹29,482
CWC Score:         0.3708
Interval Score:    29,482

TUBE LOSS BASELINE: FLAT
--------------------------------------------------
PICP (Coverage):   0.9608
MPIW (Width):      ₹15,384,809
CWC Score:         0.8097
Interval Score:    16,831,892

TUBE LOSS BASELINE: 

# Recalibration

In [167]:
def run_recalibrated_tube_loss(split_dir="split_embeddings", epochs=15, DELTA_VAL = 0.00):
    print("Recalibration of delta...\n")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    category_t_map = {
        "BIKES": 0.95, "BOOKS": 0.98, "CARS": 0.96, "CYCLE": 0.95,
        "FLAT": 0.96, "FRIDGES": 0.96, "GAMES": 0.96, "GAMESENTERTAINMENT": 0.98,
        "LAPTOP": 0.95, "PHONES": 0.98, "PRINTER": 0.98,
        "TV": 0.99, "WASHINGMACHINE": 0.98
    }
    
    train_files = [f for f in os.listdir(os.path.join(split_dir, 'train')) if f.endswith('.pt')]
    
    for file in train_files:
        cat_name = file.replace('combined_', '').replace('.pt', '').upper()
        train_path = os.path.join(split_dir, 'train', file)
        test_path = os.path.join(split_dir, 'test', file)
        
        if not os.path.exists(test_path): continue
            
        t_val = category_t_map.get(cat_name, 0.95)
        alpha = 1.0 - t_val
        
        print(f"TUBE LOSS BASELINE: {cat_name}\n{'-'*50}")
        
        # Load Data
        raw_train_data = torch.load(train_path, weights_only=False)
        cat_mu = raw_train_data['log_prices'].mean().item()
        cat_sigma = raw_train_data['log_prices'].std().item()
        
        train_dataset = ScaledRFPDataset(train_path, cat_mu, cat_sigma)
        test_dataset = ScaledRFPDataset(test_path, cat_mu, cat_sigma)
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
        
        # Initialize
        model = SimpleCategoryMLP().to(device)
        optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        # Train (DELTA = 0.0 for initial training as per paper)
        model.train()
        for epoch in range(epochs):
            for x, y in train_loader:
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad()
                out = model(x)
                loss = tube_loss(y, out[:,0], out[:,1], t=t_val, r=0.5, delta=0.0)
                loss.backward()
                optimizer.step()

        # Evaluate
        model.eval()
        lowers, uppers, ys = [], [], []
        
        with torch.no_grad():
            for x_batch, y_batch in test_loader:
                x_device = x_batch.to(device)
                out = model(x_device)
                l, u = torch.min(out[:,0], out[:,1]), torch.max(out[:,0], out[:,1])
                
                l_real = torch.exp(l * cat_sigma + cat_mu)
                u_real = torch.exp(u * cat_sigma + cat_mu)
                y_true = torch.exp(y_batch.to(device) * cat_sigma + cat_mu)
                
                lowers.append(l_real); uppers.append(u_real); ys.append(y_true)
                
        l_preds = torch.cat(lowers)
        u_preds = torch.cat(uppers)
        y_trues = torch.cat(ys)
        
        picp = ((y_trues >= l_preds) & (y_trues <= u_preds)).float().mean().item()
        mpiw = (u_preds - l_preds).mean().item()
        is_score = calculate_interval_score(y_trues, l_preds, u_preds, alpha=alpha)
        cwc_score = calculate_cwc(picp, mpiw, y_trues, t_val=t_val)
        
        print(f"PICP (Coverage):   {picp:.4f}")
        print(f"MPIW (Width):      ₹{mpiw:,.0f}")
        print(f"CWC Score:         {cwc_score:.4f}")
        print(f"Interval Score:    {is_score:,.0f}\n")

torch.manual_seed(42)

In [168]:
print("Delta:0.00")
run_recalibrated_tube_loss(epochs=15,DELTA_VAL=0.00)

Delta:0.00
Recalibration of delta...

TUBE LOSS BASELINE: BIKES
--------------------------------------------------
PICP (Coverage):   1.0000
MPIW (Width):      ₹235,194
CWC Score:         0.3733
Interval Score:    235,194

TUBE LOSS BASELINE: BOOKS
--------------------------------------------------
PICP (Coverage):   0.9565
MPIW (Width):      ₹7,146
CWC Score:         3.7136
Interval Score:    36,025

TUBE LOSS BASELINE: CARS
--------------------------------------------------
PICP (Coverage):   1.0000
MPIW (Width):      ₹6,996,750
CWC Score:         2.2746
Interval Score:    6,996,750

TUBE LOSS BASELINE: CYCLE
--------------------------------------------------
PICP (Coverage):   0.9655
MPIW (Width):      ₹446,230
CWC Score:         5.6130
Interval Score:    446,677

TUBE LOSS BASELINE: FLAT
--------------------------------------------------
PICP (Coverage):   0.9804
MPIW (Width):      ₹21,371,280
CWC Score:         1.1248
Interval Score:    24,895,990

TUBE LOSS BASELINE: FRIDGES
----

In [169]:
print("Delta:0.01")
run_recalibrated_tube_loss(epochs=15,DELTA_VAL=0.01)

Delta:0.01
Recalibration of delta...

TUBE LOSS BASELINE: BIKES
--------------------------------------------------
PICP (Coverage):   1.0000
MPIW (Width):      ₹331,379
CWC Score:         0.5260
Interval Score:    331,379

TUBE LOSS BASELINE: BOOKS
--------------------------------------------------
PICP (Coverage):   0.8478
MPIW (Width):      ₹6,349
CWC Score:         741.9416
Interval Score:    51,852

TUBE LOSS BASELINE: CARS
--------------------------------------------------
PICP (Coverage):   0.9355
MPIW (Width):      ₹21,771,068
CWC Score:         10.4846
Interval Score:    21,864,452

TUBE LOSS BASELINE: CYCLE
--------------------------------------------------
PICP (Coverage):   1.0000
MPIW (Width):      ₹29,482
CWC Score:         0.3708
Interval Score:    29,482

TUBE LOSS BASELINE: FLAT
--------------------------------------------------
PICP (Coverage):   0.9608
MPIW (Width):      ₹15,384,809
CWC Score:         0.8097
Interval Score:    16,831,892

TUBE LOSS BASELINE: FRIDGES
-

In [170]:
print("Delta:0.02")
run_recalibrated_tube_loss(epochs=15,DELTA_VAL=0.02)

Delta:0.02
Recalibration of delta...

TUBE LOSS BASELINE: BIKES
--------------------------------------------------
PICP (Coverage):   1.0000
MPIW (Width):      ₹231,333
CWC Score:         0.3672
Interval Score:    231,333

TUBE LOSS BASELINE: BOOKS
--------------------------------------------------
PICP (Coverage):   0.9783
MPIW (Width):      ₹11,969
CWC Score:         1.8931
Interval Score:    12,024

TUBE LOSS BASELINE: CARS
--------------------------------------------------
PICP (Coverage):   0.6452
MPIW (Width):      ₹5,042,825
CWC Score:         6864931.9466
Interval Score:    6,731,370

TUBE LOSS BASELINE: CYCLE
--------------------------------------------------
PICP (Coverage):   0.9310
MPIW (Width):      ₹18,941
CWC Score:         2.8195
Interval Score:    49,208

TUBE LOSS BASELINE: FLAT
--------------------------------------------------
PICP (Coverage):   0.9804
MPIW (Width):      ₹28,705,814
CWC Score:         1.5108
Interval Score:    30,516,192

TUBE LOSS BASELINE: FRIDGES

In [171]:
print("Delta:0.03")
run_recalibrated_tube_loss(epochs=15,DELTA_VAL=0.03)

Delta:0.03
Recalibration of delta...

TUBE LOSS BASELINE: BIKES
--------------------------------------------------
PICP (Coverage):   1.0000
MPIW (Width):      ₹254,995
CWC Score:         0.4048
Interval Score:    254,995

TUBE LOSS BASELINE: BOOKS
--------------------------------------------------
PICP (Coverage):   1.0000
MPIW (Width):      ₹33,476
CWC Score:         2.2437
Interval Score:    33,476

TUBE LOSS BASELINE: CARS
--------------------------------------------------
PICP (Coverage):   0.7419
MPIW (Width):      ₹6,802,894
CWC Score:         54353.5991
Interval Score:    7,692,710

TUBE LOSS BASELINE: CYCLE
--------------------------------------------------
PICP (Coverage):   0.9310
MPIW (Width):      ₹36,309
CWC Score:         3.0380
Interval Score:    53,903

TUBE LOSS BASELINE: FLAT
--------------------------------------------------
PICP (Coverage):   0.8824
MPIW (Width):      ₹12,499,258
CWC Score:         49.1961
Interval Score:    28,638,670

TUBE LOSS BASELINE: FRIDGES


In [172]:
print("Delta:0.04")
run_recalibrated_tube_loss(epochs=15,DELTA_VAL=0.04)

Delta:0.04
Recalibration of delta...

TUBE LOSS BASELINE: BIKES
--------------------------------------------------
PICP (Coverage):   0.9773
MPIW (Width):      ₹223,153
CWC Score:         0.3542
Interval Score:    250,317

TUBE LOSS BASELINE: BOOKS
--------------------------------------------------
PICP (Coverage):   0.9130
MPIW (Width):      ₹6,874
CWC Score:         28.9015
Interval Score:    29,613

TUBE LOSS BASELINE: CARS
--------------------------------------------------
PICP (Coverage):   0.7097
MPIW (Width):      ₹4,260,018
CWC Score:         272702.0724
Interval Score:    5,586,386

TUBE LOSS BASELINE: CYCLE
--------------------------------------------------
PICP (Coverage):   0.8966
MPIW (Width):      ₹28,786
CWC Score:         14.8369
Interval Score:    43,707

TUBE LOSS BASELINE: FLAT
--------------------------------------------------
PICP (Coverage):   0.9804
MPIW (Width):      ₹15,980,424
CWC Score:         0.8411
Interval Score:    19,220,040

TUBE LOSS BASELINE: FRIDGES

In [173]:
print("Delta:0.05")
run_recalibrated_tube_loss(epochs=15,DELTA_VAL=0.05)

Delta:0.05
Recalibration of delta...

TUBE LOSS BASELINE: BIKES
--------------------------------------------------
PICP (Coverage):   0.9318
MPIW (Width):      ₹163,088
CWC Score:         2.7409
Interval Score:    304,945

TUBE LOSS BASELINE: BOOKS
--------------------------------------------------
PICP (Coverage):   0.8913
MPIW (Width):      ₹6,625
CWC Score:         84.7777
Interval Score:    41,845

TUBE LOSS BASELINE: CARS
--------------------------------------------------
PICP (Coverage):   0.8065
MPIW (Width):      ₹5,838,753
CWC Score:         2160.9384
Interval Score:    6,301,315

TUBE LOSS BASELINE: CYCLE
--------------------------------------------------
PICP (Coverage):   0.9655
MPIW (Width):      ₹27,994
CWC Score:         0.3521
Interval Score:    28,252

TUBE LOSS BASELINE: FLAT
--------------------------------------------------
PICP (Coverage):   0.9804
MPIW (Width):      ₹19,084,268
CWC Score:         1.0044
Interval Score:    22,713,294

TUBE LOSS BASELINE: FRIDGES
--

# Ensemble

In [174]:
import os
import torch
import numpy as np
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import warnings

warnings.filterwarnings('ignore')

class ScaledRFPDataset(Dataset):
    def __init__(self, pt_file_path, mu, sigma):
        data = torch.load(pt_file_path, weights_only=False)
        self.embeddings = data['embeddings']
        self.indices = data.get('dataframe_indices', list(range(len(self.embeddings)))) 
        self.targets = (data['log_prices'] - mu) / sigma
        
    def __len__(self): 
        return len(self.embeddings)
        
    def __getitem__(self, idx):
        return self.embeddings[idx], torch.tensor(self.targets[idx], dtype=torch.float32)

def tube_loss(y, mu1, mu2, t=0.95, r=0.5, delta=0.0):
    lower = torch.min(mu1, mu2)
    upper = torch.max(mu1, mu2)
    
    loss = torch.zeros_like(y)
    loss[y > upper] = t * (y[y > upper] - upper[y > upper])
    loss[y < lower] = t * (lower[y < lower] - y[y < lower])

    mid = r * upper + (1 - r) * lower
    mask = (y >= lower) & (y <= upper)
    loss[mask & (y >= mid)] = (1 - t) * (upper[mask & (y >= mid)] - y[mask & (y >= mid)])
    loss[mask & (y < mid)]  = (1 - t) * (y[mask & (y < mid)] - lower[mask & (y < mid)])
    
    base_loss = loss.mean()
    width_penalty = torch.abs(upper - lower).mean()
    
    return base_loss + (delta * width_penalty)

def calculate_interval_score(y_true, lower, upper, alpha):
    width = upper - lower
    penalty_lower = torch.where(y_true < lower, (2.0 / alpha) * (lower - y_true), torch.zeros_like(width))
    penalty_upper = torch.where(y_true > upper, (2.0 / alpha) * (y_true - upper), torch.zeros_like(width))
    return (width + penalty_lower + penalty_upper).mean().item()

def calculate_cwc(picp, mpiw, y_trues, target_t, eta=50):
    y_max, y_min = y_trues.max().item(), y_trues.min().item()
    range_y = y_max - y_min if (y_max - y_min) > 0 else 1.0
    nmpiw = mpiw / range_y
    gamma = 1 if picp < target_t else 0
    penalty = gamma * np.exp(-eta * (picp - target_t))
    return nmpiw + penalty

class PaperBaselineMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1536, 512), nn.LayerNorm(512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, 2)
        )
    def forward(self, x): 
        return self.net(x.float())

class CrossAttentionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.img_to_txt_attn = nn.MultiheadAttention(embed_dim=768, num_heads=4, batch_first=True)
        self.txt_to_img_attn = nn.MultiheadAttention(embed_dim=768, num_heads=4, batch_first=True)
        
        self.head = nn.Sequential(
            nn.Linear(1536, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(512, 128), nn.GELU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        img_emb = x[:, :768].float().unsqueeze(1) 
        txt_emb = x[:, 768:].float().unsqueeze(1)
        
        img_attended, _ = self.img_to_txt_attn(query=img_emb, key=txt_emb, value=txt_emb)
        txt_attended, _ = self.txt_to_img_attn(query=txt_emb, key=img_emb, value=img_emb)
        
        img_attended = img_attended.squeeze(1)
        txt_attended = txt_attended.squeeze(1)
        
        fused = torch.cat([img_attended, txt_attended], dim=1)
        return self.head(fused)

class GatedMultimodalUnit(nn.Module):
    def __init__(self):
        super().__init__()
        self.img_proj = nn.Linear(768, 512)
        self.txt_proj = nn.Linear(768, 512)
        self.gate = nn.Linear(1536, 512)
        self.head = nn.Sequential(
            nn.Linear(512, 128), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(128, 2)
        )
    def forward(self, x):
        img_emb, txt_emb = x[:, :768].float(), x[:, 768:].float()
        
        img_feat = torch.tanh(self.img_proj(img_emb))
        txt_feat = torch.tanh(self.txt_proj(txt_emb))
        
        z = torch.sigmoid(self.gate(x.float()))
        
        fused = (z * img_feat) + ((1 - z) * txt_feat)
        return self.head(fused)

class LateFusionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.img_net = nn.Sequential(nn.Linear(768, 256), nn.GELU(), nn.LayerNorm(256))
        self.txt_net = nn.Sequential(nn.Linear(768, 256), nn.GELU(), nn.LayerNorm(256))
        self.head = nn.Sequential(
            nn.Linear(512, 128), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(128, 2)
        )
    def forward(self, x):
        img_emb, txt_emb = x[:, :768].float(), x[:, 768:].float()
        img_feat = self.img_net(img_emb)
        txt_feat = self.txt_net(txt_emb)
        
        fused = torch.cat([img_feat, txt_feat], dim=1)
        return self.head(fused)

def evaluate_heterogeneous_ensemble(models_list, test_loader, cat_sigma, cat_mu, device, t_val, alpha):
    for m in models_list: 
        m.eval()
    
    ensemble_lowers, ensemble_uppers, ys = [], [], []
    
    with torch.no_grad():
        for x, y_batch in test_loader:
            x_device = x.to(device)
            y_true = torch.exp(y_batch.to(device) * cat_sigma + cat_mu)
            
            batch_lowers = []
            batch_uppers = []
            
            for model in models_list:
                out = model(x_device)
                l, u = torch.min(out[:,0], out[:,1]), torch.max(out[:,0], out[:,1])
                batch_lowers.append(l)
                batch_uppers.append(u)
                
            avg_l = torch.stack(batch_lowers).mean(dim=0)
            avg_u = torch.stack(batch_uppers).mean(dim=0)
            
            l_real = torch.exp(avg_l * cat_sigma + cat_mu)
            u_real = torch.exp(avg_u * cat_sigma + cat_mu)
            
            ensemble_lowers.append(l_real)
            ensemble_uppers.append(u_real)
            ys.append(y_true)
            
    l_preds = torch.cat(ensemble_lowers)
    u_preds = torch.cat(ensemble_uppers)
    y_trues = torch.cat(ys)
    
    picp = ((y_trues >= l_preds) & (y_trues <= u_preds)).float().mean().item()
    mpiw = (u_preds - l_preds).mean().item()
    is_score = calculate_interval_score(y_trues, l_preds, u_preds, alpha=alpha)
    cwc_score = calculate_cwc(picp, mpiw, y_trues, target_t=t_val)
    
    print(f"\n{'*'*50}")
    print(f"FINAL PROPOSED ENSEMBLE METRICS")
    print(f"{'*'*50}")
    print(f"PICP: {picp:.4f}")
    print(f"MPIW: {mpiw:,.0f}")
    print(f"CWC Score: {cwc_score:.4f}")
    print(f"Interval Score: {is_score:,.0f}\n")

def run_heterogeneous_ensemble(split_dir="split_embeddings", epochs=15):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    DELTA_VAL = 0.00
    
    category_t_map = {
        "BIKES": 0.95, "BOOKS": 0.98, "CARS": 0.96, "CYCLE": 0.95,
        "FLAT": 0.96, "FRIDGES": 0.96, "GAMES": 0.96, "GAMESENTERTAINMENT": 0.98,
        "LAPTOP": 0.95, "PHONES": 0.98, "PRINTER": 0.98,
        "TV": 0.99, "WASHINGMACHINE": 0.98
    }
    
    train_files = [f for f in os.listdir(os.path.join(split_dir, 'train')) if f.endswith('.pt')]
    
    for file in train_files:
        cat_name = file.replace('combined_', '').replace('.pt', '').upper()
        
        train_path = os.path.join(split_dir, 'train', file)
        test_path = os.path.join(split_dir, 'test', file)
        
        if not os.path.exists(test_path): continue
            
        target_t = category_target_map.get(cat_name, 0.95)
        alpha = 1.0 - target_t
        
        print(f"TRAINING HETEROGENEOUS ENSEMBLE: {cat_name}\n{'-'*50}")
        
        raw_train_data = torch.load(train_path, weights_only=False)
        cat_mu = raw_train_data['log_prices'].mean().item()
        cat_sigma = raw_train_data['log_prices'].std().item()
        
        train_dataset = ScaledRFPDataset(train_path, cat_mu, cat_sigma)
        test_dataset = ScaledRFPDataset(test_path, cat_mu, cat_sigma)
        
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
        
        models_to_train = [
            ("Baseline MLP", PaperBaselineMLP()),
            ("Cross-Attention", CrossAttentionNet()),
            ("GMU", GatedMultimodalUnit()),
            ("Late Fusion", LateFusionNet())
        ]
        
        trained_models = []
        
        for model_name, model_inst in models_to_train:
            print(f"Training Architecture: {model_name}...")
            model = model_inst.to(device)
            optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
            
            model.train()
            for epoch in range(epochs):
                for x, y in train_loader:
                    x, y = x.to(device), y.to(device)
                    optimizer.zero_grad()
                    out = model(x)
                    loss = tube_loss(y, out[:,0], out[:,1], t=target_t, r=0.5, delta=DELTA_VAL)
                    loss.backward()
                    optimizer.step()
                    
            trained_models.append(model)
            
        evaluate_heterogeneous_ensemble(
            trained_models, test_loader, cat_sigma, cat_mu, device, target_t, alpha
        )


In [175]:
run_heterogeneous_ensemble(epochs=15)

NameError: name 'category_target_map' is not defined

In [ ]:
import os
import torch
import numpy as np
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import random
import warnings

warnings.filterwarnings('ignore')

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

class ScaledRFPDataset(Dataset):
    def __init__(self, pt_file_path, mu, sigma):
        data = torch.load(pt_file_path, weights_only=False)
        self.embeddings = data['embeddings']
        self.indices = data.get('dataframe_indices', list(range(len(self.embeddings)))) 
        self.targets = (data['log_prices'] - mu) / sigma
        
    def __len__(self): 
        return len(self.embeddings)
        
    def __getitem__(self, idx):
        return self.embeddings[idx], torch.tensor(self.targets[idx], dtype=torch.float32)

def tube_loss(y, mu1, mu2, t=0.95, r=0.5, delta=0.0):
    lower = torch.min(mu1, mu2)
    upper = torch.max(mu1, mu2)
    loss = torch.zeros_like(y)
    loss[y > upper] = t * (y[y > upper] - upper[y > upper])
    loss[y < lower] = t * (lower[y < lower] - y[y < lower])
    mid = r * upper + (1 - r) * lower
    mask = (y >= lower) & (y <= upper)
    loss[mask & (y >= mid)] = (1 - t) * (upper[mask & (y >= mid)] - y[mask & (y >= mid)])
    loss[mask & (y < mid)]  = (1 - t) * (y[mask & (y < mid)] - lower[mask & (y < mid)])
    return loss.mean() + (delta * torch.abs(upper - lower).mean())

def calculate_interval_score(y_true, lower, upper, alpha):
    width = upper - lower
    penalty_lower = torch.where(y_true < lower, (2.0 / alpha) * (lower - y_true), torch.zeros_like(width))
    penalty_upper = torch.where(y_true > upper, (2.0 / alpha) * (y_true - upper), torch.zeros_like(width))
    return (width + penalty_lower + penalty_upper).mean().item()

def calculate_cwc(picp, mpiw, y_trues, target_t, eta=50):
    y_max, y_min = y_trues.max().item(), y_trues.min().item()
    range_y = y_max - y_min if (y_max - y_min) > 0 else 1.0
    nmpiw = mpiw / range_y
    gamma = 1 if picp < target_t else 0
    return nmpiw + (gamma * np.exp(-eta * (picp - target_t)))

class PaperBaselineMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1536, 512), nn.LayerNorm(512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, 2)
        )
    def forward(self, x): 
        return self.net(x.float())

class CrossAttentionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.img_to_txt_attn = nn.MultiheadAttention(embed_dim=768, num_heads=4, batch_first=True)
        self.txt_to_img_attn = nn.MultiheadAttention(embed_dim=768, num_heads=4, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(1536, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(512, 128), nn.GELU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        img_emb = x[:, :768].float().unsqueeze(1) 
        txt_emb = x[:, 768:].float().unsqueeze(1)
        img_attended, _ = self.img_to_txt_attn(query=img_emb, key=txt_emb, value=txt_emb)
        txt_attended, _ = self.txt_to_img_attn(query=txt_emb, key=img_emb, value=img_emb)
        fused = torch.cat([img_attended.squeeze(1), txt_attended.squeeze(1)], dim=1)
        return self.head(fused)

def evaluate_mixed_ensemble(models_list, test_loader, cat_sigma, cat_mu, device, t_val, alpha):
    for m in models_list: 
        m.eval()
    ensemble_lowers, ensemble_uppers, ys = [], [], []
    with torch.no_grad():
        for x, y_batch in test_loader:
            x_device = x.to(device)
            y_true = torch.exp(y_batch.to(device) * cat_sigma + cat_mu)
            batch_lowers, batch_uppers = [], []
            for model in models_list:
                out = model(x_device)
                l, u = torch.min(out[:,0], out[:,1]), torch.max(out[:,0], out[:,1])
                batch_lowers.append(l)
                batch_uppers.append(u)
            avg_l = torch.stack(batch_lowers).mean(dim=0)
            avg_u = torch.stack(batch_uppers).mean(dim=0)
            ensemble_lowers.append(torch.exp(avg_l * cat_sigma + cat_mu))
            ensemble_uppers.append(torch.exp(avg_u * cat_sigma + cat_mu))
            ys.append(y_true)
            
    l_preds = torch.cat(ensemble_lowers)
    u_preds = torch.cat(ensemble_uppers)
    y_trues = torch.cat(ys)
    
    picp = ((y_trues >= l_preds) & (y_trues <= u_preds)).float().mean().item()
    mpiw = (u_preds - l_preds).mean().item()
    is_score = calculate_interval_score(y_trues, l_preds, u_preds, alpha=alpha)
    cwc_score = calculate_cwc(picp, mpiw, y_trues, target_t=t_val)
    
    print(f"\n{'*'*50}")
    print(f"FINAL MIXED ENSEMBLE METRICS")
    print(f"{'*'*50}")
    print(f"PICP: {picp:.4f}")
    print(f"MPIW: {mpiw:,.0f}")
    print(f"CWC Score: {cwc_score:.4f}")
    print(f"Interval Score: {is_score:,.0f}\n")

def run_mixed_ensemble(split_dir="split_embeddings", n_baseline=3, epochs=15,DELTA_VAL = 0.00):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    category_t_map = {
        "BIKES": 0.95, "BOOKS": 0.98, "CARS": 0.97, "CYCLE": 0.95,
        "FLAT": 0.96, "FRIDGES": 0.97, "GAMES": 0.96, "GAMESENTERTAINMENT": 0.99,
        "LAPTOP": 0.95, "PHONES": 0.97, "PRINTER": 0.98,
        "TV": 0.99, "WASHINGMACHINE": 0.98
    }
    
    train_files = [f for f in os.listdir(os.path.join(split_dir, 'train')) if f.endswith('.pt')]
    for file in train_files:
        cat_name = file.replace('combined_', '').replace('.pt', '').upper()
        train_path = os.path.join(split_dir, 'train', file)
        test_path = os.path.join(split_dir, 'test', file)
        if not os.path.exists(test_path): continue
            
        target_t = category_t_map.get(cat_name, 0.95)
        alpha = 1.0 - target_t
        print(f"\n{'='*70}\nTRAINING MIXED ENSEMBLE: {cat_name}\n{'='*70}")
        
        raw_train_data = torch.load(train_path, weights_only=False)
        cat_mu = raw_train_data['log_prices'].mean().item()
        cat_sigma = raw_train_data['log_prices'].std().item()
        train_dataset = ScaledRFPDataset(train_path, cat_mu, cat_sigma)
        test_dataset = ScaledRFPDataset(test_path, cat_mu, cat_sigma)
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
        
        trained_models = []
        for i in range(n_baseline):
            current_seed = 42 + i
            set_seed(current_seed)
            print(f"Training Baseline MLP {i + 1}/{n_baseline} (Seed {current_seed})...")
            model = PaperBaselineMLP().to(device)
            optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
            model.train()
            for epoch in range(epochs):
                for x, y in train_loader:
                    x, y = x.to(device), y.to(device)
                    optimizer.zero_grad()
                    out = model(x)
                    loss = tube_loss(y, out[:,0], out[:,1], t=target_t, r=0.5, delta=DELTA_VAL)
                    loss.backward()
                    optimizer.step()
            trained_models.append(model)
            
        cross_attn_seed = 42 + n_baseline
        set_seed(cross_attn_seed)
        print(f"Training Cross-Attention Model (Seed {cross_attn_seed})...")
        ca_model = CrossAttentionNet().to(device)
        ca_optimizer = AdamW(ca_model.parameters(), lr=1e-3, weight_decay=1e-4)
        ca_model.train()
        for epoch in range(epochs):
            for x, y in train_loader:
                x, y = x.to(device), y.to(device)
                ca_optimizer.zero_grad()
                out = ca_model(x)
                loss = tube_loss(y, out[:,0], out[:,1], t=target_t, r=0.5, delta=DELTA_VAL)
                loss.backward()
                ca_optimizer.step()
        trained_models.append(ca_model)
        
        evaluate_mixed_ensemble(trained_models, test_loader, cat_sigma, cat_mu, device, target_t, alpha)



In [ ]:
run_mixed_ensemble(n_baseline=3, epochs=15)


TRAINING MIXED ENSEMBLE: BIKES
Training Baseline MLP 1/3 (Seed 42)...
Training Baseline MLP 2/3 (Seed 43)...
Training Baseline MLP 3/3 (Seed 44)...
Training Cross-Attention Model (Seed 45)...

**************************************************
FINAL MIXED ENSEMBLE METRICS
**************************************************
PICP: 1.0000
MPIW: 281,790
CWC Score: 0.4473
Interval Score: 281,790


TRAINING MIXED ENSEMBLE: BOOKS
Training Baseline MLP 1/3 (Seed 42)...
Training Baseline MLP 2/3 (Seed 43)...
Training Baseline MLP 3/3 (Seed 44)...
Training Cross-Attention Model (Seed 45)...

**************************************************
FINAL MIXED ENSEMBLE METRICS
**************************************************
PICP: 0.9565
MPIW: 8,307
CWC Score: 3.7914
Interval Score: 29,865


TRAINING MIXED ENSEMBLE: CARS
Training Baseline MLP 1/3 (Seed 42)...
Training Baseline MLP 2/3 (Seed 43)...
Training Baseline MLP 3/3 (Seed 44)...
Training Cross-Attention Model (Seed 45)...

********************

KeyboardInterrupt: 